# Análise de redes de depedências de pacotes Python

In [24]:
import pandas as pd
import requests
from json import load
from operator import itemgetter
import networkx as nx

from pyvis.network import Network
from matplotlib import cm
from matplotlib.colors import rgb2hex

## Download requirements

In [25]:
with open("C:\\Users\\angel\\OneDrive\\Documentos\\MBA\\TCC\\Dataset\\Downloads\\full_scam.json", "r") as f:
	download_ranking = pd.DataFrame(load(f))

In [26]:
deps = []
for index, row in download_ranking.head(100).iterrows():
	current_deps = {}
	response = requests.get(f"https://api.deps.dev/v3/systems/pypi/packages/{row['package_name']}/versions/{row['version']}")
	if response.status_code != 200:
		continue
	is_default, is_deprecated, attestations = itemgetter("isDefault", "isDeprecated", "attestations")(response.json())
	is_verified = False
	if attestations:
		attestation, = attestations
		is_verified = attestation.get("verified", False)
	current_deps |= {
		"package_name": row["package_name"],
		"version": row["version"],
		"is_default": is_default,
		"is_deprecated": is_deprecated,
		"is_verified": is_verified
	}
	response = requests.get(f"https://api.deps.dev/v3/systems/pypi/packages/{row['package_name']}/versions/{row['version']}:requirements")
	if response.status_code != 200:
		continue
	requirements = response.json()["pypi"]["dependencies"]
	current_deps |= {
		"requirements": requirements
	}
	deps.append(current_deps)

In [27]:
df_deps = pd.DataFrame(deps)

In [28]:
df_deps = df_deps.explode("requirements")

In [29]:
df_deps =  pd.concat([df_deps.drop('requirements', axis=1), df_deps['requirements'].apply(pd.Series)], axis=1)

In [30]:
df_deps = df_deps.rename(columns={'package_name': 'source', 'projectName': 'target'}).dropna(subset=['target', 'source'])

## Grafos

In [31]:
edges = df_deps[["source", "target"]].values.tolist()


In [32]:
G = nx.DiGraph()

In [33]:
G.add_edges_from(edges)

In [35]:
pagerank = nx.pagerank(G)

In [36]:
betweenness = nx.betweenness_centrality(G)

In [37]:
degree = nx.degree_centrality(G)

In [38]:
weakly_connected_components = list(nx.weakly_connected_components(G))

In [41]:
net = Network(directed=True, notebook=True)
net.from_nx(G)

net.show_buttons()
net.write_html("output\\dependency_graph.html")